## Requirements to run this notebook standalone

Place these two files in the same folder as this notebook before running:
- `smartcare_ai_dataset_1000.csv`
- `smartcare_ai_dataset_data_dictionary.csv`

This notebook covers **Task 02 (Dataset Understanding)** and **Task 03 (Data Preprocessing and Feature Engineering)**. Run all cells top to bottom. It produces the files used by the later task notebooks: `X_train.csv`, `X_test.csv`, `X_train_scaled.csv`, `X_test_scaled.csv`, `y_train.csv`, `y_test.csv`, and `model_df.csv`.

## SmartCare Hospital AI Dataset — Patient Readmission Prediction (30 days)

This notebook covers **Task 02 (Dataset Understanding)** and **Task 03 (Data Preprocessing and Feature Engineering)**:

**Task 02**
1. Explore the dataset
2. Identify input and target variables
3. Explain attribute meanings using data dictionary
4. Discuss data quality

**Task 03**
1. Missing value handling
2. Duplicate record detection
3. Outlier identification
4. Data cleaning
5. Feature engineering
6. Feature selection
7. Feature encoding
8. Feature scaling
9. Train/test split and export of the model-ready dataset

# **Task 02 — Dataset Understanding**

## 2.1. Import Required Libraries

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

ModuleNotFoundError: No module named 'pandas'

## 2.2. Import the Dataset

In [ ]:
df1 = pd.read_csv('../data/raw/smartcare_ai_dataset_1000.csv')
data_dict = pd.read_csv('../data/raw/smartcare_ai_dataset_data_dictionary.csv')
print(df1.shape)
df1.head(10)

In [ ]:
print("\nColumn Names:")
print(df1.columns)

## 2.3. Datset Overview

In [ ]:
df1.info()

In [ ]:
df1.describe(include='all').T

## 2.4. **Summer of the data overview**

1,000 rows, 33 columns where one row describes one patient appointment.

**record_id** and **patient_id** are unique row identifiers.

Columns span five natural groups:

##### **demographics**

*   age
*   gender
*   blood_group

##### **appointment/admission history**

*   department
*   diagnosis
*   appointment_date
*   waiting_days
*   previouse_appointment
*   missed_previous_appointments
*   appointment_status
*   admitted
*   room_type
*   length_of_stay_days
*   previous_admissions

##### **clinical measurements**

*   systolic_bp
*   diastolic_bp
*   blood_sugar_mg_dl
*   cholesterol_mg_dl
*   bmi
*   lab_tests_count
*   treatments_count

##### **Financial data**

*   consultation_fee_lkr
*   room_charge_lkr
*   lab_charge_lkr
*   medicine_charge_lkr
*   total_bill_lkr
*   payment_status
*   payment_method

##### **prediction targets**

*   no_show
*   readmitted_30_days
*   disease_risk_level



**Selected prediction target - readmitted_30_days**

## 2.5. Checking for the Distribution of Readmitted Prediction

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(x='readmitted_30_days', data=df1)

plt.title('Distribution of Readmission Prediction')
plt.xlabel('Readmission Prediction')
plt.ylabel('Number of Patients')
plt.show()

In [ ]:
class_counts = df1['readmitted_30_days'].value_counts()
class_percentages = df1['readmitted_30_days'].value_counts(normalize=True) * 100

print("Class Counts:")
print(class_counts)

print("\nClass Percentages:")
print(class_percentages)

imbalance_ratio = class_counts.min() / class_counts.max()

print(f"\nImbalance Ratio: {imbalance_ratio:.2f}")

This is a imbalanced dataset because in readmitted_30_days column, for yes(1) there are only 25.3% and for not admitted (0) there are 74.7% records.

## 2.6. Attribute Description Using Data Dictionary

In [ ]:
attr_desc = data_dict.copy()
attr_desc['dtype'] = attr_desc['Column'].map(df1.dtypes.astype(str))
attr_desc

## 2.7. Identifying Input and Target Variables

- readmitted_30_days is used as the target variable (y) for this project.

In [ ]:
target_col = 'readmitted_30_days'
excluded_targets = ['no_show', 'disease_risk_level']
id_cols = ['record_id', 'patient_id']

input_cols = [c for c in df1.columns if c not in [target_col] + excluded_targets + id_cols]
print(f'Target variable: {target_col}')
print(f'Excluded alternate targets: {excluded_targets}')
print(f'ID columns (dropped, not predictive): {id_cols}')
print(f'\nNumber of candidate input variables: {len(input_cols)}')
print(input_cols)

## 2.8. Data Quality Discussion

Before deciding how to clean and preprocess the data we have to quantify and identify data qualty issues in this raw dataset.

### 2.8.1. Logical Consistency Check

In [ ]:
print('Logical consistency checks')
neg_charges = (df1[['consultation_fee_lkr','room_charge_lkr','lab_charge_lkr',
                    'medicine_charge_lkr','total_bill_lkr']] < 0).any().any()
print('Any negative charges?', neg_charges)

bill_check = (df1['consultation_fee_lkr'] + df1['room_charge_lkr'] + df1['lab_charge_lkr']
              + df1['medicine_charge_lkr'])
print('Rows where total_bill_lkr matches sum of components:',
      (bill_check == df1['total_bill_lkr']).sum(), '/', len(df1))

missed_gt_prev = (df1['missed_previous_appointments'] > df1['previous_appointments']).sum()
print('Rows where missed_previous_appointments > previous_appointments (impossible):', missed_gt_prev)

illogical_admit = df1[(df1['admitted'] == 1) & (df1['appointment_status'].isin(['No-Show', 'Cancelled']))]
print('Rows marked admitted=1 despite a No-Show/Cancelled appointment_status (illogical):', len(illogical_admit))

los_nonadmit = ((df1['admitted'] == 0) & (df1['length_of_stay_days'] != 0)).sum()
print('Non-admitted patients with nonzero length_of_stay_days:', los_nonadmit)

### 2.8.2. Identifying Missing Values

In [ ]:
print('--- Missing values ---')
missing = df1.isnull().sum()
missing = missing[missing > 0]
print(missing)
print('\nAs % of rows:')
print((missing / len(df1) * 100).round(1))

In [ ]:
# 1. Missing room_type breakdown by admitted status
print(df1['room_type'].isna().groupby(df1['admitted']).sum())

# 2. Subgroup: admitted==1 but room_type missing AND room_charge_lkr==0
subgroup = df1[(df1['admitted'] == 1) & (df1['room_type'].isna()) & (df1['room_charge_lkr'] == 0)]

print("Rows in subgroup:", len(subgroup))

# 3. Their average length of stay (should be ≈ 3.3 days)
print("Mean length_of_stay_days:", subgroup['length_of_stay_days'].mean())

# 4. Their readmission rate vs overall dataset average
print("Subgroup readmission rate:", subgroup['readmitted_30_days'].mean())
print("Overall readmission rate:", df1['readmitted_30_days'].mean())

Only **room_type** has missing values. This is not random. Because it is almost entirely missing for **admitted == 0** patients (who logically never occupy a room)

But also 236 rows are **admitted == 1** with a missing **room_type** and **room_charge_lkr == 0**.

# **Task 03 — Data Preprocessing and Feature Engineering**

## 3.1. Get a copy of the Original Dataset

In [ ]:
df = df1.copy()

print("Original dataset shape:", df.shape)

## 3.2. Missing Value Handling

In [ ]:
df['room_type'].isnull().sum()

In [ ]:
df['room_type'].value_counts(dropna=False)

In [ ]:
missing_room = df[df['room_type'].isnull()]

missing_room[['length_of_stay_days','room_charge_lkr','readmitted_30_days']].describe()

In [ ]:
df[
    (df['room_type'].isnull()) &
    (df['length_of_stay_days'] == 0) &
    (df['room_charge_lkr'] == 0)
]

In [ ]:
pd.crosstab(df['admitted'], df['room_type'].isna(), rownames=['admitted'], colnames=['room_type is missing'])

In [ ]:
df.isnull().sum().sum()

In [ ]:
df['room_type'] = df['room_type'].fillna('Unknown')
df.loc[df['admitted'] == 0, 'room_type'] = 'Not Admitted'
df['room_type'].value_counts(dropna=False)

In [ ]:
pd.crosstab(df['readmitted_30_days'], df['admitted'], rownames=['readmitted_30_days'], colnames=['admitted'])

In [ ]:
df.isnull().sum().sum()

If, room_type = NaN, length_of_stay_days == 0 and, room_charge_lkr == 0 we have, those records has been replaced as **Not Admitted**.

If room_type = NaN, lenth_of_stay_days > 0, it is replaced as **Unknown**.

## 3.3. Duplicate Record Detection

In [ ]:
print('Fully duplicated rows:', df.duplicated().sum())
print('Duplicated patient_id values:', df.duplicated(subset=['patient_id']).sum())
print('Duplicated record_id values:', df.duplicated(subset=['record_id']).sum())

There are no duplicate records in this dataset. So no rows are dropped at this step.

## 3.4. Outlier Identification

In [ ]:
numeric_cols = ['age', 'waiting_days', 'previous_appointments', 'missed_previous_appointments',
                'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp',
                'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count',
                'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr',
                'total_bill_lkr']

fig, axes = plt.subplots(4, 5, figsize=(20, 14))
axes = axes.flatten()
for i, col in enumerate(numeric_cols):
    sns.boxplot(y=df[col], ax=axes[i], color='#4C72B0')
    axes[i].set_title(col, fontsize=10)
for j in range(len(numeric_cols), len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.savefig('../outputs/outlier_boxplots.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
def iqr_outlier_summary(data, cols):
    rows = []
    for col in cols:
        q1, q3 = data[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((data[col] < lower) | (data[col] > upper)).sum()
        rows.append({'column': col, 'lower_bound': round(lower, 1), 'upper_bound': round(upper, 1),
                     'n_outliers': n_out, 'pct_outliers': round(n_out / len(data) * 100, 2)})
    return pd.DataFrame(rows).sort_values('pct_outliers', ascending=False)

outlier_summary = iqr_outlier_summary(df, numeric_cols)
outlier_summary

In [ ]:
print('Age range:', df['age'].min(), '-', df['age'].max())
print('BMI range:', df['bmi'].min(), '-', df['bmi'].max())
print('Any negative charges?', (df[['consultation_fee_lkr','room_charge_lkr','lab_charge_lkr',
                                     'medicine_charge_lkr','total_bill_lkr']] < 0).any().any())
print('Any negative/zero length_of_stay for admitted patients?',
      ((df['admitted']==1) & (df['length_of_stay_days'] <= 0)).sum())

the IQR-flagged points sit within clinically or financially plausible ranges. For example the
highest **total_bill_lkr** values occurs becuase of genuinely long ICU stays, not data entry errors, and
**age/bmi** stay within realistic human bounds.

Because they represent real or normal values than impossible or error values, **no outliers are removed or capped**. Removing them would discard
exactly the kind of high-cost, high-risk patients a readmission model most needs to learn from.

The IQR summary is kept as a reference table so the modeling stage is aware of which features are skewed.

## 3.5. Data Cleaning

In [ ]:
bill_check = (df['consultation_fee_lkr'] + df['room_charge_lkr'] + df['lab_charge_lkr']
              + df['medicine_charge_lkr'])
print('Rows where total_bill_lkr matches sum of components:', (bill_check == df['total_bill_lkr']).sum(), '/', len(df))

In [ ]:
# missed_previous_appointments cannot exceed previous_appointments.
bad_mask = df['missed_previous_appointments'] > df['previous_appointments']
print('Rows fixed (missed > previous):', bad_mask.sum())
df.loc[bad_mask, 'missed_previous_appointments'] = df.loc[bad_mask, 'previous_appointments']

In [ ]:
# a patient cannot be 'admitted' if their appointment was never attended
# (appointment_status of 'No-Show' or 'Cancelled'). Flag rather than silently drop, since we
# cannot be certain which of the two fields is the error; a flag lets the model / analyst account
# for the inconsistency without losing the row.
df['admission_status_conflict'] = (
    (df['admitted'] == 1) & (df['appointment_status'].isin(['No-Show', 'Cancelled']))
).astype(int)
print('Rows flagged with an admission/appointment_status conflict:', df['admission_status_conflict'].sum())

In [ ]:
df['appointment_date'] = pd.to_datetime(df['appointment_date'])
df['appointment_month'] = df['appointment_date'].dt.month
df['appointment_dayofweek'] = df['appointment_date'].dt.dayofweek  # 0=Mon

cat_cols_to_clean = ['gender', 'blood_group', 'department', 'diagnosis', 'appointment_status',
                     'room_type', 'payment_status', 'payment_method']
for c in cat_cols_to_clean:
    df[c] = df[c].astype(str).str.strip()

df[cat_cols_to_clean].nunique()

**"missed_previous_appointments" capped at "previous_appointments"**

- This means that if a patient's data says they missed more appointments than they ever had in total (which is impossible), we fix it by setting the missed count equal to the total count. We fixed the number instead of deleting the whole row, so we don't lose data unnecessarily.


**"Admitted"/"appointment_status" conflicts are flagged, not deleted**

- For the 186 rows where a patient is marked "admitted" but also marked as a "No-Show" or "Cancelled", we don't know which of the two fields is wrong. Maybe "admitted" was mistakenly set to 1, or maybe the appointment status was logged wrong. Since we can't tell which one is the mistake, instead of guessing, we just add a new colums(admission_status_conflict) that marks these rows as **"suspicious"**. This way, the row stays in the dataset.

- **Limitation:** this flag turns out to correlate strongly with the target (see Task 04 EDA / SHAP analysis in Task 07). That correlation should be treated with caution — it may reflect the model picking up on a genuine clinical/administrative inconsistency, but it may equally be an artefact of how the synthetic dataset was generated (e.g. conflicting rows being more likely to also carry `readmitted_30_days = 1` by construction, rather than by any real clinical mechanism). Its use as a predictive feature is kept, but this should be flagged as an open question in the report rather than presented as a confirmed clinical finding.


**Categorical text columns are stripped of whitespace**

- Sometimes text data has hidden extra spaces, like "Male " (with a trailing space) instead of "Male". Since "Male " and "Male" are two different categories, if left uncleaned, this creates fake duplicate categories when we later convert text into numbers (one-hot encoding). Stripping whitespace prevents this silent duplication.


**"appointment_date" is parsed to datetime and decomposed into "appointment_month" and "appointment_dayofweek"**

- The raw date like "2025-08-08" isn't very useful to a model as just a text string — it can't learn patterns from it directly. But parts of the date might matter in cases like maybe more patients get readmitted in certain months (seasonal illness patterns), or certain days of the week have different appointment behavior. So instead of keeping the full date, we break it into two more useful, model-friendly pieces such as the month and the day of the week.

## 3.6. Identification and Flagging of Zero Room Charges

In [ ]:
condition = (df['length_of_stay_days'] > 0) & (df['room_charge_lkr'] == 0)

df['zero_room_charge_flag'] = condition.astype(int)

print("Affected records:", condition.sum())
print("Percentage:", round(condition.mean() * 100, 2), "%")

display(df.loc[condition, [
    'length_of_stay_days',
    'room_charge_lkr',
    'room_type',
    'zero_room_charge_flag'
]].head())

Records with **"length_of_stay_days"** > 0 and **"room_charge_lkr"** = 0 may indicate either a genuinely zero or a no-charge stay. It might also indicate missing or unrecorded billing information.

Since the exact reason cannot be determined from the available dataset, these records were neither removed nor modified. Instead, a binary **"zero_room_charge_flag"** was created to preserve the original information and allow the model to consider this pattern as a potential predictive feature for readmission.

## 3.7. Feature Engineering

In [ ]:
df['total_prior_visits'] = df['previous_appointments'] + df['previous_admissions']

df['missed_appointment_rate'] = np.where(
    df['previous_appointments'] > 0,
    df['missed_previous_appointments'] / df['previous_appointments'],
    0
)

def bmi_band(b):
    if b < 18.5: return 'Underweight'
    if b < 25:   return 'Normal'
    if b < 30:   return 'Overweight'
    return 'Obese'
df['bmi_category'] = df['bmi'].apply(bmi_band)

def bp_band(row):
    sys_, dia_ = row['systolic_bp'], row['diastolic_bp']
    if sys_ < 120 and dia_ < 80:
        return 'Normal'
    if sys_ < 140 and dia_ < 90:
        return 'Elevated'
    return 'Hypertensive'
df['bp_category'] = df.apply(bp_band, axis=1)

util_q75 = (df['lab_tests_count'] + df['treatments_count']).quantile(0.75)
df['high_utilisation'] = ((df['lab_tests_count'] + df['treatments_count']) >= util_q75).astype(int)

df['avg_charge_per_treatment'] = df['total_bill_lkr'] / (df['treatments_count'] + 1)

df['is_weekend_appointment'] = df['appointment_dayofweek'].isin([5, 6]).astype(int)

df[['total_prior_visits', 'missed_appointment_rate', 'bmi_category', 'bp_category',
    'high_utilisation', 'avg_charge_per_treatment', 'is_weekend_appointment']].head()

**Justification for each engineered feature:**
- **"total_prior_visits"** — combines outpatient and inpatient history into one measure of overall prior
  healthcare utilisation, which is a well-known driver of readmission risk.

- **"missed_appointment_rate"** — a *rate* is more comparable across patients than a raw count.

- **"bmi_category"** / **"bp_category"** — convert continuous clinical measurements into clinically meaningful
  bands (standard medical thresholds), which can capture non-linear risk effects that a linear model on
  the raw number would miss, and are easier to interpret.

- **"high_utilisation"** — a simple binary flag for "top quartile" of lab tests + treatments, useful as a
  cheap proxy for how clinically intensive a visit was.

- **"avg_charge_per_treatment"** — This line creates a new feature: average cost per treatment, calculated as total_bill_lkr / (treatments_count + 1).
The +1 just avoids a division-by-zero error for patients with 0 treatments.
Why it's useful: total_bill_lkr alone
can be misleading. A patient might have a huge bill simply because they stayed longer (more room/lab charges), not because their treatments were expensive. Dividing by treatment count removes that "length of stay" effect and isolates how costly each treatment was on average.
The benefit: this gives the model a cleaner signal of which "cost intensity per treatment", instead of a number dominated by how many days someone stayed.

- **"is_weekend_appointment"** — weekend appointments may reflect emergency/urgent care patterns with
  different readmission risk than routine weekday visits.

In [ ]:
df.columns.tolist()

## 3.8. Feature Selection

In [ ]:
drop_cols = ['record_id', 'patient_id', 'appointment_date', 'no_show', 'disease_risk_level',
             'consultation_fee_lkr', 'room_charge_lkr', 'lab_charge_lkr', 'medicine_charge_lkr']

model_df = df.drop(columns=drop_cols)
print(model_df.shape)
model_df.columns.tolist()

- **"record_id"**, **"patient_id"** — They identify the patient/record but don't provide meaningful predictive information.

- **"appointment_date"** — decomposed into **"appointment_month"** / **"appointment_dayofweek"**, which
  capture the reusable signal in the date without the model memorising specific calendar dates.

- **"no_show"**, **"disease_risk_level"** — targets for the other two prediction tasks in this dataset.

- **"consultation_fee_lkr"**, **"room_charge_lkr"**, **"lab_charge_lkr"**, **"medicine_charge_lkr"** — Because they sum to **"total_bill_lkr"**, making them perfectly collinear with it. Keeping **"total_bill_lkr"** alone keeps the same financial information without duplications.

In [ ]:
num_features = model_df.select_dtypes(include=[np.number]).drop(columns=['readmitted_30_days'])
corr = num_features.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=0.4)
plt.title('Correlation Heatmap — Numeric Features')
plt.tight_layout()
plt.savefig('../outputs/correlation_heatmap.png', dpi=110, bbox_inches='tight')
plt.show()

In [ ]:
high_corr_pairs = (corr.abs()
                    .where(np.triu(np.ones(corr.shape), k=1).astype(bool))
                    .stack()
                    .sort_values(ascending=False))
high_corr_pairs[high_corr_pairs > 0.6]

In [ ]:
target_corr = model_df.select_dtypes(include=[np.number]).corr()['readmitted_30_days'].drop('readmitted_30_days')
target_corr.sort_values(key=abs, ascending=False)

A correlation heatmap and a list of feature pairs with correlation above 0.6 were used for two purposes:

(1) to check if any other numeric features are still redundant with each other, beyond the four charge columns already removed

(2) to see which numeric features have the strongest linear relationship with the target variable — useful context before modeling.

**Result**:

No remaining feature pairs exceeded the 0.6 correlation threshold, confirming that removing the four charge columns was enough to eliminate the major redundancy in the numeric data.

**Why no more columns were dropped**:

Moderate correlation (below 0.6) isn't a strong enough reason to remove a feature-tree-based models (like Random Forest, XGBoost) aren't affected by correlated features at all, and regularized linear models (like Ridge/Lasso Logistic Regression) can handle moderate correlation without becoming unstable. Since this is still an early, exploratory stage, dropping more columns now risks throwing away genuinely useful signal for a very small, uncertain benefit.

## 3.9. Feature Encoding

In [ ]:
bmi_order = {'Underweight': 0, 'Normal': 1, 'Overweight': 2, 'Obese': 3}
bp_order = {'Normal': 0, 'Elevated': 1, 'Hypertensive': 2}

model_df['bmi_category'] = model_df['bmi_category'].map(bmi_order)
model_df['bp_category'] = model_df['bp_category'].map(bp_order)

print('Ordinal encoding check:')
print(model_df[['bmi_category', 'bp_category']].dtypes)
print(model_df[['bmi_category', 'bp_category']].head())

In [ ]:
nominal_cols = ['gender', 'blood_group', 'department', 'diagnosis',
                'appointment_status', 'room_type', 'payment_status', 'payment_method']

model_df_encoded = pd.get_dummies(model_df, columns=nominal_cols, drop_first=True)

print('Shape after one-hot encoding:', model_df_encoded.shape)
model_df_encoded.head()

In [ ]:
# Step 3: Convert True/False dummy columns to 0/1 integers (safer for scikit-learn/XGBoost)
bool_cols = model_df_encoded.select_dtypes(include='bool').columns
model_df_encoded[bool_cols] = model_df_encoded[bool_cols].astype(int)

print('Final encoded dataset shape:', model_df_encoded.shape)
print('\nColumn dtypes summary:')
print(model_df_encoded.dtypes.value_counts())

## 3.10. Train/Test Split and Feature scaling

In [ ]:
X = model_df_encoded.drop(columns=['readmitted_30_days', 'high_utilisation'])
y = model_df_encoded['readmitted_30_days']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Recompute high_utilisation without leakage: threshold derived from
# training data only, then applied unchanged to the test set.
util_q75 = (X_train['lab_tests_count'] + X_train['treatments_count']).quantile(0.75)

X_train['high_utilisation'] = ((X_train['lab_tests_count'] + X_train['treatments_count']) >= util_q75).astype(int)
X_test['high_utilisation'] = ((X_test['lab_tests_count'] + X_test['treatments_count']) >= util_q75).astype(int)

print('high_utilisation threshold (train-only q75):', util_q75)

print('Train shape:', X_train.shape, ' Test shape:', X_test.shape)
print('Train class balance:\n', y_train.value_counts(normalize=True).round(3))
print('Test class balance:\n', y_test.value_counts(normalize=True).round(3))

## 3.10.1. Save Test-Set Demographics for Fairness Assessment (Task 07)

`X_train`/`X_test` are saved later with `index=False`, which discards which original patient each row belongs to. Since `model_df` (pre-encoding) still has readable `gender`/`age`/`department` columns aligned to the same row order as `X_test` (train_test_split preserves the original index, and no reordering happens between here and export), this is the point to capture them — needed for the fairness/subgroup analysis in Task 07, not the model itself.

In [ ]:
test_demographics = model_df.loc[X_test.index, ['gender', 'age', 'department']].copy()

test_demographics['age_group'] = pd.cut(
    test_demographics['age'],
    bins=[0, 30, 50, 70, 120],
    labels=['<30', '30-49', '50-69', '70+']
)

test_demographics.to_csv('../data/processed/test_demographics.csv', index=False)
print('Saved test_demographics.csv, shape:', test_demographics.shape)
test_demographics.head()

**"high_utilisation"** was originally computed on the full dataset (Task 02 EDA version), which risked test-set leakage since the quartile threshold would be informed by test rows. For the modelling pipeline, this was corrected by deriving the threshold from the training set only and applying it unchanged to the test set.

The split is stratified on **"readmitted_30_days"** so the train and test sets both
preserve the same ~25% positive-class rate seen in the raw data (see Data Quality discussion), which
matters for a moderately imbalanced target. The split is done **before** scaling so that the scaler is
fit only on training data in the next step, avoiding test-set information leaking into the transformation.

In [ ]:
continuous_cols = ['age', 'waiting_days', 'previous_appointments', 'missed_previous_appointments',
                   'length_of_stay_days', 'previous_admissions', 'systolic_bp', 'diastolic_bp',
                   'blood_sugar_mg_dl', 'cholesterol_mg_dl', 'bmi', 'lab_tests_count', 'treatments_count',
                   'total_bill_lkr', 'total_prior_visits', 'missed_appointment_rate',
                   'avg_charge_per_treatment', 'appointment_month', 'appointment_dayofweek']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

X_train_scaled[continuous_cols].describe().T[['mean', 'std']].round(2)

only genuinely continuous/count columns are scaled; the one-hot dummy columns and
binary flags ("admitted", "high_utilisation", "is_weekend_appointment", "admission_status_conflict") are
deliberately left unscaled since they are already on a meaningful 0/1 scale and scaling them would distort
that meaning. "StandardScaler" (zero mean, unit variance) is used rather than min-max scaling because
several continuous features (e.g. "total_bill_lkr", "avg_charge_per_treatment") are right-skewed with
legitimate high-value outliers (retained per the Outlier Identification decision above); standardisation
is less sensitive to those extreme values than a min-max range. The scaler is **fit on the training set
only** and applied to the test set with `.transform()`, preventing test-set leakage into the scaling
parameters.

## 3.11. Export preprocessed data for EDA

In [ ]:
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
X_train_scaled.to_csv('../data/processed/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/processed/X_test_scaled.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Files saved:')
for f in ['X_train.csv','X_test.csv','X_train_scaled.csv','X_test_scaled.csv','y_train.csv','y_test.csv']:
    print(' -', '../data/processed',f)

In [ ]:
import joblib

joblib.dump(scaler, '../models/scaler.pkl')
joblib.dump(model_df_encoded.drop(columns=['readmitted_30_days']).columns.tolist(), '../models/model_columns.pkl')

joblib.dump(continuous_cols, '../models/scaled_columns.pkl')
joblib.dump(nominal_cols, '../models/nominal_cols.pkl')

joblib.dump(util_q75, '../models/high_utilisation_threshold.pkl')


## 3.12. Export **"model_df"** for Task 04 (EDA)

Saved separately so Task 04 can be run as its own notebook without re-running Tasks 02-03.

In [ ]:
model_df.to_csv('../data/processed/model_df.csv', index=False)
print('Saved model_df.csv, shape:', model_df.shape)